In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/CryAndRRich/codapath.git"
REPO_BRANCH = "namhai"
REPO = Path("/kaggle/working/codapath")

if (REPO / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "switch", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH])
elif REPO.exists():
    raise RuntimeError(f"{REPO} exists but is not a Git repository")
else:
    subprocess.check_call(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO)]
    )

branch = subprocess.check_output(
    ["git", "-C", str(REPO), "branch", "--show-current"], text=True
).strip()
assert branch == REPO_BRANCH, (branch, REPO_BRANCH)
print("repo:", REPO, "| branch:", branch)

In [ ]:
%cd /kaggle/working/codapath

In [ ]:
import os
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U",
                       "huggingface_hub<1.0", "hf-transfer"])

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
if "/kaggle/working/codapath" not in sys.path:
    sys.path.append("/kaggle/working/codapath")

In [ ]:
DATASET = "pathmnist"  # pathmnist | histoset | skintissue
SEED = 42  # any int; one seed per run

IMAGE_ENCODER = "dinov2"  # dinov2 | conch
USE_TEXT = False  # True | False -- True needs IMAGE_ENCODER="conch"
DESCRIPTION_STYLE = "llm_short"  # llm_short | llm_morphology -- only read when USE_TEXT
CELL_POOLING = "mean"  # mean | rff | moments

SEL_AUX_WEIGHT = 0.0  # 0.0 = off | 1 | 5 | 20

USE_LORA = False  # True | False
LORA_R = 8  # 2 | 4 | 8 -- ignored when USE_LORA=False
LORA_LR = 1e-4  # 1e-4 | 1e-5 -- ignored when USE_LORA=False
AUX_LOSS = "none"  # none | center | supcon | triplet -- needs USE_LORA=True
AUGMENT = "none"  # none | flip_rotate

HF_TOKEN = ""  # "" = fall back to Kaggle Secret HF_TOKEN

OVERRIDES = {}  # {} = config.yaml as-is; e.g. {'uncertainty_mode': 'visual_margin'}
if CELL_POOLING != "mean":
    OVERRIDES = {**OVERRIDES, "cell_pooling": CELL_POOLING}
if SEL_AUX_WEIGHT > 0.0:
    OVERRIDES = {
        **OVERRIDES,
        "pool_consistency_weight": SEL_AUX_WEIGHT,
    }

RUN_NAME = None  # None = derive from sampler config

PARALLEL = True  # True | False

SPLIT_BUDGETS = True  # True | False

FEATURE_DIR = "/kaggle/input/datasets/nhtquyn/pathoactive"
CELLVIT_DIR = "/kaggle/input/datasets/cryandrrich/pathoactivenucleus"
VLM_FEATURE_DIR = "/kaggle/input/datasets/cryandrrich/nckh2026/vlm_features"
OUTPUT_DIR = "/kaggle/working/checkpoints"

MMAP_CACHE_DIR = "/kaggle/working/npz_mmap"

In [ ]:
import os

from training.finetune import needs_pixels

LOADS_ENCODER = needs_pixels(USE_LORA, AUGMENT, AUX_LOSS)

if IMAGE_ENCODER == "dinov2":
    from huggingface_hub import snapshot_download

    print("Downloading facebook/dinov2-base ...")
    snapshot_download(repo_id="facebook/dinov2-base")
elif not LOADS_ENCODER:
    print("IMAGE_ENCODER='conch' with no final-training pass: reads the")
    print("published feature cache only -- no checkpoint, no HF token needed.")
else:
    if not HF_TOKEN:
        try:
            from kaggle_secrets import UserSecretsClient

            HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
            print("[auth] using Kaggle Secret HF_TOKEN")
        except Exception as exc:
            print(f"[auth] no Kaggle Secret HF_TOKEN ({type(exc).__name__})")
    assert HF_TOKEN, (
        "IMAGE_ENCODER='conch' with USE_LORA/AUGMENT on loads the GATED CONCH "
        "checkpoint, which needs a Hugging Face token. Set HF_TOKEN in the EDIT "
        "cell, or add a Kaggle Secret named HF_TOKEN (Add-ons -> Secrets). "
        "Without one the download fails with a 401/403 partway into the run."
    )
    os.environ["HF_TOKEN"] = HF_TOKEN

    from huggingface_hub import login

    login(HF_TOKEN)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/mahmoodlab/CONCH.git"])
    print("conch installed; CONCH checkpoint will load in the final-training pass")

In [ ]:
import yaml
import torch

import main
from sampling.specs import spec_for
from utils.parallel import run_variants_parallel, visible_gpu_count
from utils.kaggle import find_cellvit_cache, find_data_root, find_visual_cache, find_vlm_cache
from utils.progress import format_duration

In [ ]:
DATA_ROOT = find_data_root()

DATA_PATHS = {
    "pathmnist": str(DATA_ROOT / "pathmnist_224.npz"),
    "histoset": str(DATA_ROOT / "HistoSet-5x14/HistoSet-5x14"),
    "skintissue": str(DATA_ROOT / "SkinTissue/SkinTissue/tiles"),
}
print("data root:", DATA_ROOT)

In [ ]:
with open("config/config.yaml", "r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

dataset_info = config["datasets"][DATASET]
training_cfg = config["training"]
SAMPLER = "scalpel"

sampler_cfg = {**config.get("samplers", {}).get(SAMPLER, {}), **OVERRIDES}
spec = spec_for(SAMPLER)

data_path = Path(DATA_PATHS[DATASET])
assert data_path.exists(), f"Missing Kaggle input: {data_path}"
assert torch.cuda.is_available(), "Attach a Kaggle GPU before running AL"
assert isinstance(SEED, int), "SEED is one seed, not a list -- re-run the notebook to sweep"
assert isinstance(OVERRIDES, dict), "OVERRIDES is one config dict, not a list of variants"
assert SAMPLER == "scalpel", (
    "run_al_main.ipynb always runs scalpel; a baseline goes through "
    "run_al_baseline.ipynb instead"
)
assert IMAGE_ENCODER in ("dinov2", "conch"), f"Unknown IMAGE_ENCODER={IMAGE_ENCODER!r}"

if USE_TEXT:
    # Contribution #1: the round-1 cold start.
    #
    # Round 1 has no labels, so scalpel's two probes cannot exist and the round
    # is plain MaxHerding -- coverage with no notion of which regions are hard.
    # With a text tower, each patch is scored against one prototype per class
    # and weighted by the margin of that zero-shot distribution: the SAME
    # "near a decision boundary" quantity rounds 2+ use, sourced without a
    # single label.
    assert IMAGE_ENCODER == "conch", (
        "USE_TEXT needs a text tower in the same space as the image tower; "
        "DINOv2 has none. Set IMAGE_ENCODER='conch'."
    )
    assert SAMPLER == "scalpel", (
        f"USE_TEXT is an axis of scalpel's round 1; {SAMPLER!r} has no such "
        "round. Leave USE_TEXT=False for a baseline."
    )

assert AUX_LOSS in ("none", "center", "supcon", "triplet"), f"Unknown AUX_LOSS={AUX_LOSS!r}"
assert AUGMENT in ("none", "flip_rotate"), f"Unknown AUGMENT={AUGMENT!r}"
if AUX_LOSS != "none":
    assert USE_LORA, (
        f"AUX_LOSS={AUX_LOSS!r} needs USE_LORA=True. The auxiliary losses act "
        "on encoder features, which have no gradient while the encoder is "
        "frozen, so the term would contribute nothing and the run would "
        "duplicate the AUX_LOSS='none' baseline under a different name."
    )

FINAL_TRAIN_CFG = {
    "use_lora": USE_LORA, "lora_r": LORA_R,
    "lora_alpha": 2.0 * LORA_R,
    "lora_lr": LORA_LR,
    "aux_loss": AUX_LOSS, "aux_weight": 0.5, "augment": AUGMENT,
}

if "cell_embeddings" in spec.needs:
    found = find_cellvit_cache(DATASET, SEED, hint=CELLVIT_DIR)
    assert found is not None, (
        f"No CellViT cache for {DATASET}_seed{SEED} under {CELLVIT_DIR} or "
        f"the default Kaggle input roots. Attach the extraction dataset."
    )
    CELLVIT_DIR = str(found)
    print("cellvit cache:", CELLVIT_DIR)
    assert not (IMAGE_ENCODER == "conch" and sampler_cfg.get("cell_source") == "crop_dino"), (
        "cell_source='crop_dino' was extracted with DINOv2; using it with "
        "IMAGE_ENCODER='conch' mixes two incompatible image spaces. Use "
        "cell_source='cellvit_embedding' instead."
    )

if IMAGE_ENCODER == "conch":
    vlm_name = config.get("models", {}).get("vlm", "MahmoodLab/CONCH")
    found = find_vlm_cache(DATASET, SEED, vlm_name, hint=VLM_FEATURE_DIR)
    assert found is not None, (
        f"No CONCH feature cache for {DATASET}_seed{SEED}_{vlm_name} under "
        f"{VLM_FEATURE_DIR} or the default Kaggle input roots. Run "
        f"extract_vlm_features.ipynb first and attach its published dataset."
    )
    VLM_FEATURE_DIR = str(found)
    print("vlm cache:", VLM_FEATURE_DIR)
else:
    vit_name = config.get("models", {}).get("vit", "facebook/dinov2-base")
    found = find_visual_cache(DATASET, SEED, vit_name, hint=FEATURE_DIR)
    if found is not None:
        FEATURE_DIR = str(found)
        print("features cache:", FEATURE_DIR)
    else:
        print(f"[features] not found for {DATASET}/seed{SEED}/{vit_name} - will extract this session.")
        print("  (a .npy with no matching manifest is rejected too -- row alignment")
        print("  cannot be verified without it.)")
        FEATURE_DIR = "/kaggle/working/features"
    if not str(FEATURE_DIR).startswith("/kaggle/input"):
        Path(FEATURE_DIR).mkdir(parents=True, exist_ok=True)

SAVE_DIR = Path(OUTPUT_DIR) / DATASET
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"{SAMPLER}: {spec.passes} pass, prefix_exact={spec.prefix_exact} - {spec.why}")
print(f"image_encoder: {IMAGE_ENCODER}")
print(f"final_train_cfg: {FINAL_TRAIN_CFG}")
print(f"GPUs visible: {visible_gpu_count()}")
print("config:", sampler_cfg)

In [ ]:
import time

BUDGETS = config["cumulative_budget"]
workers = visible_gpu_count() if PARALLEL else 1

shard_budgets = SPLIT_BUDGETS and workers > 1 and not spec.prefix_exact and len(BUDGETS) > 1
if SPLIT_BUDGETS and spec.prefix_exact:
    print(f"[shard] {SAMPLER} is prefix-exact: one shared selection pass covers every")
    print("        budget, so its sweep stays on a single GPU.")

def budget_shards(budgets, n):
    """Round-robin, not contiguous: cost grows with the budget, so a contiguous
    split would hand one worker every large budget."""
    groups = [budgets[i::n] for i in range(n)]
    return [g for g in groups if g]

RUN = RUN_NAME or main._default_run_name(
    SAMPLER, sampler_cfg, encoder=IMAGE_ENCODER, final_train_cfg=FINAL_TRAIN_CFG,
    use_text=USE_TEXT, description_style=DESCRIPTION_STYLE,
)
if SEED != config.get("random_seed", 42):
    RUN = f"{RUN}_s{SEED}"

base_kwargs = dict(
    data_path=str(data_path),
    sampler_name=SAMPLER,
    num_classes=dataset_info["num_classes"],
    data_descriptions=dataset_info.get("descriptions", {}),
    prompt_templates=config.get("prompt_templates", []),
    sampler_cfg=sampler_cfg,
    probe_epochs=training_cfg["probe_epochs"],
    probe_lr=training_cfg["probe_lr"],
    random_seed=SEED,
    save_dir=str(SAVE_DIR),
    verbose=True,
    model_cfg=config.get("models", {}),
    feature_cache_dir=FEATURE_DIR,
    mmap_cache_dir=MMAP_CACHE_DIR,
    cellvit_cache_dir=CELLVIT_DIR,
    run_name=RUN,
    device_string="cuda:0",
    image_encoder=IMAGE_ENCODER,
    vlm_cache_dir=VLM_FEATURE_DIR,
    use_text=USE_TEXT,
    description_style=DESCRIPTION_STYLE,
    hf_token=(HF_TOKEN or None) if (IMAGE_ENCODER == "conch" and LOADS_ENCODER) else None,
    final_train_cfg=FINAL_TRAIN_CFG,
)

jobs = []
shard_tags = []
if (SAVE_DIR / f"{RUN}_results.pt").is_file():
    print(f"already finished, nothing to do: {RUN}_results.pt")
elif shard_budgets:
    shards = budget_shards(BUDGETS, workers)
    shard_tags = [f"shard{i}" for i in range(len(shards))]
    for tag, budgets in zip(shard_tags, shards):
        jobs.append((f"{RUN}:{tag}", dict(
            base_kwargs, cumulative_budget=budgets, shard_tag=tag,
        )))
else:
    jobs.append((RUN, dict(base_kwargs, cumulative_budget=BUDGETS)))

print(f"run: {RUN}")
for label, kwargs in jobs:
    print(f"   {label:40} budgets={kwargs['cumulative_budget']}")

if str(data_path).endswith(".npz"):
    from data.npz_mmap import export_npz_to_npy

    export_npz_to_npy(str(data_path), MMAP_CACHE_DIR)
    print(f"mmap export ready: {MMAP_CACHE_DIR}")
else:
    print("ImageFolder dataset: reads per file, no mmap export needed")

started = time.time()
results = run_variants_parallel(jobs, main.run_on_worker, num_workers=workers)

print("=" * 70)
for result in results:
    status = "ok" if result["ok"] else "FAILED"
    print(f"{result['label']:40} {status:8} {format_duration(result['seconds'])}")
failed = [r["label"] for r in results if not r["ok"]]
if results:
    print(f"total {format_duration(time.time() - started)} | "
          f"{len(results) - len(failed)}/{len(results)} succeeded")
assert not failed, f"jobs failed: {failed}"

if shard_tags:
    linear = main.merge_budget_shards(str(SAVE_DIR), RUN, shard_tags)
    print(f"[merge] {RUN}: {len(linear)} budgets -> {RUN}_results.pt")

In [ ]:
import torch

results_path = SAVE_DIR / f"{RUN}_results.pt"
assert results_path.is_file(), (
    f"no results at {results_path} -- the run above did not finish"
)
payload = torch.load(results_path, weights_only=False)
linear = payload["linear"]

print(f"{payload['sampler']}  |  {payload['dataset']}  |  seed {payload['seed']}")
print(f"run_name: {payload['run_name']}")
if payload.get("sharded_over"):
    print(f"budget shards: {', '.join(payload['sharded_over'])}")
print()

header = f"{'budget':>8}  {'accuracy':>9}  {'precision':>9}  {'recall':>9}  {'macro F1':>9}  {'select s':>9}"
print(header)
print("-" * len(header))
for budget in sorted(linear):
    row = linear[budget]
    print(f"{budget:>8}  {row['acc']:>9.4f}  {row['precision']:>9.4f}  "
          f"{row['recall']:>9.4f}  {row['f1']:>9.4f}  {row['selection_seconds']:>9.1f}")
print("-" * len(header))

best = max(linear, key=lambda b: linear[b]["acc"])
print(f"best accuracy {linear[best]['acc']:.4f} at budget {best}")

worst = {b: linear[b].get("sanity_severity", "ok") for b in sorted(linear)}
flagged = {b: s for b, s in worst.items() if s != "ok"}
if flagged:
    print(f"sanity: {flagged}  <- check the run log for details")
else:
    print("sanity: ok at every budget")

In [ ]:
import shutil

from utils import main_archive_stem

if MMAP_CACHE_DIR and Path(MMAP_CACHE_DIR).is_dir():
    freed = sum(f.stat().st_size for f in Path(MMAP_CACHE_DIR).rglob("*") if f.is_file())
    shutil.rmtree(MMAP_CACHE_DIR, ignore_errors=True)
    print(f"removed mmap export, freed {freed / 2**30:.1f} GiB")
SOURCE = Path(OUTPUT_DIR)
WORKING = Path("/kaggle/working")
assert SOURCE.is_dir(), f"nothing to archive at {SOURCE}"
assert SOURCE.resolve() != WORKING.resolve(), (
    "OUTPUT_DIR must be a subdirectory of /kaggle/working, not /kaggle/working itself"
)

STEM = main_archive_stem(DATASET, SAMPLER, SEED, encoder=IMAGE_ENCODER, run_name=RUN)
ARCHIVE = WORKING / STEM
shutil.make_archive(str(ARCHIVE), "zip", root_dir=SOURCE)
size_mb = ARCHIVE.with_suffix(".zip").stat().st_size / 1e6

print(f"{ARCHIVE.name}.zip  ({size_mb:.1f} MB) contains:")
for path in sorted(SOURCE.rglob("*")):
    if path.is_file():
        print(f"    {path.relative_to(SOURCE)}  ({path.stat().st_size / 1e6:.2f} MB)")

shutil.rmtree(SOURCE, ignore_errors=True)

remaining = sorted(p for p in WORKING.iterdir() if p.name != "codapath")
total_mb = sum(
    f.stat().st_size for p in remaining for f in ([p] if p.is_file() else p.rglob("*"))
    if f.is_file()
) / 1e6
print(f"\n/kaggle/working now holds {total_mb:.1f} MB (Output quota ~20 GB):")
for path in remaining:
    print(f"    {path.name}{'/' if path.is_dir() else ''}")

print(f"""
NEXT STEPS (no terminal needed)
  1. Output tab (right panel) -> download {ARCHIVE.name}.zip
  2. kaggle.com/datasets -> New Dataset -> upload that zip
  3. In evaluate_al_sampler.ipynb: Add Data -> your new dataset, then point
     CHECKPOINT_ROOT at it to rebuild the table and fit PALM.""")